# Exotica `.0000.fil` — real GBT data exploration

First look at the real Breakthrough Listen Exotica narrowband product (`.rawspec.0000.h5`), now that a batch has arrived. Purely descriptive: inventory the batch, look at cadence structure, and visualize a handful of representative files raw and preprocessed. This notebook does **not** judge or revise the `configs/data/gbt_fine.yaml` assumptions — that's a separate step once this batch has been looked at.

Run this notebook on the machine where the data lives (synced via git, executed remotely).

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "scripts"))

from scan_headers import (
    parse_filename,
    read_header,
    cluster_by_gap,
    infer_on_off,
)
from src.data.preprocessing import bandpass_correct, core_transform

In [ ]:
# --- Config ---------------------------------------------------------------
# Server path: /home/acabras/data/wd_mybook
# Container path (this is what the notebook should see when run on the remote host): /content/wd_mybook
DATA_DIR = Path("/content/wd_mybook")
PRODUCT = "0000"
GAP_THRESHOLD_S = 700  # inter-cadence gap, per scan_headers.py default

OUT_CSV = REPO_ROOT / "data" / "processed" / "exotica_0000_headers.csv"
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)

## 1. Batch inventory

Scan every `.0000.h5` file under `DATA_DIR`, parse the filename, and read the header (no data load) using the same logic `scripts/scan_headers.py` uses for the CLI report.

In [ ]:
files = sorted(DATA_DIR.rglob(f"*.{PRODUCT}.h5"))
print(f"Found {len(files)} .{PRODUCT}.h5 file(s) under {DATA_DIR}")

rows = []
for f in files:
    parsed = parse_filename(f)
    header = read_header(f)
    row = dict(parsed)
    if "error" not in header:
        row.update(header)
    else:
        row["header_error"] = header["error"]
    rows.append(row)

df = pd.DataFrame(rows)
bad = df[df.get("parse_error").notna() | df.get("header_error").notna()] if len(df) else df
print(f"{len(bad)} file(s) with parse/header errors")
bad

In [ ]:
good = df[df["nchans"].notna()].sort_values("abs_time_s").reset_index(drop=True) if len(df) else df
good.to_csv(OUT_CSV, index=False)
print(f"Saved header inventory -> {OUT_CSV}  ({len(good)} rows)")
good[["file", "mjd_day", "scan_num", "target", "f_start_MHz", "f_stop_MHz", "bw_MHz", "nchans", "ntime", "duration_s"]]

## 2. Cadence / ON-OFF structure

Cluster by time gap and infer which target is ON (Exotica source, repeated) vs OFF (reference stars, single appearance) within each cadence.

In [ ]:
rows_sorted = good.to_dict("records")
clusters = cluster_by_gap(rows_sorted, GAP_THRESHOLD_S)
print(f"{len(clusters)} cadence(s) inferred from {len(rows_sorted)} files")

cadence_summary = []
for idx, cluster in enumerate(clusters, 1):
    oi = infer_on_off(cluster)
    times = [r["abs_time_s"] for r in cluster]
    cadence_summary.append({
        "cadence": idx,
        "n_files": len(cluster),
        "span_s": max(times) - min(times),
        "on_target": oi["on_target"],
        "off_targets": ", ".join(oi["off_targets"]),
        "pattern": oi["pattern"],
    })

cadence_df = pd.DataFrame(cadence_summary)
cadence_df

## 3. Frequency / bandwidth / duration coverage

In [ ]:
print(f"global freq range : {good['f_start_MHz'].min():.2f} - {good['f_stop_MHz'].max():.2f} MHz")
print(f"bandwidth per file : min={good['bw_MHz'].min():.2f}  max={good['bw_MHz'].max():.2f}  median={good['bw_MHz'].median():.2f} MHz")
print(f"duration per file  : min={good['duration_s'].min():.1f}  max={good['duration_s'].max():.1f}  median={good['duration_s'].median():.1f} s")
print(f"nchans unique values: {sorted(good['nchans'].unique())}")
print(f"ntime unique values : {sorted(good['ntime'].unique())}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(good["bw_MHz"], bins=20)
axes[0].set_title("Bandwidth per file (MHz)")
axes[1].hist(good["duration_s"], bins=20)
axes[1].set_title("Duration per file (s)")
plt.tight_layout()

In [ ]:
print("Unique targets observed (ON or OFF):")
print(good["target"].value_counts())

## 4. Representative files

Pick 3-5 files spanning ON and OFF roles from different cadences/targets to inspect visually.

In [ ]:
sample_files = []
for cluster in clusters:
    oi = infer_on_off(cluster)
    on_row = next((r for r in cluster if r["target"] == oi["on_target"]), None)
    off_row = next((r for r in cluster if r["target"] != oi["on_target"]), None)
    if on_row:
        sample_files.append(("ON", on_row))
    if off_row:
        sample_files.append(("OFF", off_row))
    if len(sample_files) >= 5:
        break

for role, r in sample_files:
    print(f"[{role}] {r['target']:<25} {Path(r['file']).name}")

## 5. Raw waterfalls

`.0000.h5` files are ~67M channels wide (2.79 Hz/chan) — too large to load or plot in full. Grab a narrow sub-band (a few thousand channels, comparable to the `fchans=1024` model window) around the middle of the covered band using blimpy's `f_start`/`f_stop`.

In [ ]:
import blimpy

SUBBAND_CHANNELS = 4096  # a few times the model's fchans=1024 window, for context

def load_subband(file_path, n_channels=SUBBAND_CHANNELS):
    """Load a narrow frequency window around the observation's band center."""
    wf_hdr = blimpy.Waterfall(str(file_path), load_data=False)
    fch1 = wf_hdr.header["fch1"]
    foff = wf_hdr.header["foff"]
    nchans = wf_hdr.header["nchans"]
    f_lo = min(fch1, fch1 + foff * nchans)
    f_hi = max(fch1, fch1 + foff * nchans)
    f_center = (f_lo + f_hi) / 2
    half_width = n_channels * abs(foff) / 2
    wf = blimpy.Waterfall(
        str(file_path),
        f_start=f_center - half_width,
        f_stop=f_center + half_width,
    )
    data = wf.data
    if data.ndim == 3:
        data = data[:, 0, :]
    return data.astype(np.float32)

In [ ]:
fig, axes = plt.subplots(len(sample_files), 1, figsize=(10, 3 * len(sample_files)))
if len(sample_files) == 1:
    axes = [axes]

raw_frames = {}
for ax, (role, r) in zip(axes, sample_files):
    frame = load_subband(r["file"])
    raw_frames[r["file"]] = frame
    ax.imshow(frame, aspect="auto", origin="lower", cmap="viridis")
    ax.set_title(f"[{role}] {r['target']} — {Path(r['file']).name}")
    ax.set_xlabel("channel")
    ax.set_ylabel("time bin")
plt.tight_layout()

## 6. Raw vs preprocessed

Run the same sub-bands through the pipeline's `bandpass_correct` + `core_transform` (default config: `method="polynomial"`, `poly_degree=3`) to see what the model actually sees.

In [ ]:
fig, axes = plt.subplots(len(sample_files), 2, figsize=(14, 3 * len(sample_files)))
if len(sample_files) == 1:
    axes = axes[np.newaxis, :]

for row_idx, (role, r) in enumerate(sample_files):
    frame = raw_frames[r["file"]]
    corrected = bandpass_correct(frame, method="polynomial", poly_degree=3)
    normalized = core_transform(corrected)

    axes[row_idx, 0].imshow(frame, aspect="auto", origin="lower", cmap="viridis")
    axes[row_idx, 0].set_title(f"raw [{role}] {r['target']}")

    axes[row_idx, 1].imshow(normalized, aspect="auto", origin="lower", cmap="viridis")
    axes[row_idx, 1].set_title(f"bandpass-corrected + log/MAD [{role}] {r['target']}")

plt.tight_layout()